# HW: Promptable Segmentation with the Segment Anything Model (SAM)

## Deep Learning / Computer Vision — Programming Assignment

**Big picture.** SAM is a **foundation model** for segmentation: *promptable* and
*zero-shot*. Given a point, a box, or nothing, it "cuts out" objects with **no
training on your data**. You will run SAM automatically and with prompts, then
**evaluate it against ground truth** with IoU.

**Notebook input →** a pretrained SAM checkpoint (ViT-H, ~2.4 GB, auto-downloaded),
example images, and a labeled COCO dataset (Roboflow MRI brain-tumor) for
evaluation. Paper: [arXiv:2304.02643](https://arxiv.org/abs/2304.02643).
**Notebook output →** automatic masks, prompted masks, and IoU vs ground truth.

You are expected to **read the `segment-anything` / `supervision` docs and work
out the details yourself.** Cells state only the input/output of each function.
**A GPU runtime is required.**

### Deliverables & grading (100 pts)
| Part | Topic | Pts |
|------|-------|-----|
| 1 | Load the SAM model | 10 |
| 2 | Automatic mask generation | 20 |
| 3 | Visualize / filter masks | 15 |
| 4 | Box-prompted segmentation | 20 |
| 5 | Point-prompted segmentation | 15 |
| 6 | Quantitative evaluation (IoU) | 15 |
| — | Written answers (✍️) | 5 |

> Fill in every `# TODO` and ✍️ question. Keep the provided setup/plumbing and asserts.


## Part 0 — Setup  *(provided — just run)*

In [ ]:
!nvidia-smi

In [ ]:
import os
HOME = os.getcwd()
print("HOME:", HOME)

In [ ]:
%cd {HOME}
import sys
!{sys.executable} -m pip install 'git+https://github.com/facebookresearch/segment-anything.git'
!pip install -q jupyter_bbox_widget roboflow dataclasses-json supervision

In [ ]:
%cd {HOME}
!mkdir -p {HOME}/weights
!wget -q -nc -P {HOME}/weights https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
CHECKPOINT_PATH = os.path.join(HOME, "weights", "sam_vit_h_4b8939.pth")
print(CHECKPOINT_PATH, "; exists:", os.path.isfile(CHECKPOINT_PATH))

In [ ]:
%cd {HOME}
!mkdir -p {HOME}/data
!wget -q -nc -P {HOME}/data https://media.roboflow.com/notebooks/examples/dog.jpeg
!wget -q -nc -P {HOME}/data https://media.roboflow.com/notebooks/examples/dog-2.jpeg
print("data:", os.listdir(os.path.join(HOME, "data")))

## Part 1 — Load the SAM model  *(10 pts)*

In [ ]:
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

# TODO: pick DEVICE (GPU if available), MODEL_TYPE="vit_h", and load `sam` from the
# checkpoint onto the device.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_TYPE = "vit_h"
sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH)
sam.to(device=DEVICE)

assert sam is not None
print("SAM loaded on", DEVICE)


✍️ **Q1.1** Contrast SAM's zero-shot/promptable paradigm with the trained
U-Net: what does each assume about the classes you care about?

**Your answer:** _

U-Net: fixed, closed set of classes learned from labeled training data. Only works within that distribution but gives semantic labels directly (like tumor as an example).

SAM: no class assumptions and zero-shot, promptable, works on any image domain.
SAM outputs where an object is (a mask), not what it is.

Trade-off: 

U-Net = semantics but limited generality.

SAM = generality but no semantics (we can supply meaning via the prompt)._

## Part 2 — Automatic mask generation  *(20 pts)*

In [ ]:
# TODO: mask_generator = SamAutomaticMaskGenerator(sam)
mask_generator = SamAutomaticMaskGenerator(sam)
assert mask_generator is not None


In [ ]:
import cv2
IMAGE_PATH = os.path.join(HOME, "data", "dog.jpeg")

# TODO: read the image, convert BGR->RGB (SAM expects RGB), and generate masks.
image_bgr =cv2.imread(IMAGE_PATH)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
sam_result = mask_generator.generate(image_rgb)   # -> list of mask dicts

assert isinstance(sam_result, list) and len(sam_result) > 0
print(f"SAM found {len(sam_result)} masks")


In [ ]:
# TODO: inspect the output. Print one result dict's keys, and for the first few
# masks print their area / predicted_iou / stability_score.

print("Keys in one mask dict:", sam_result[0].keys())

for i, mask in enumerate(sam_result[:5]):
    print(f"Mask {i}: area={mask['area']}, "
          f"predicted_iou={mask['predicted_iou']:.3f}, "
          f"stability_score={mask['stability_score']:.3f}")


✍️ **Q2.1** What do `predicted_iou` and `stability_score` measure, and how
can SAM "predict" an IoU with no ground truth?

✍️ **Q2.2** Automatic masks have no class names. What extra component turns SAM
into a *semantic* segmenter?

**Your answers:**

- Q2.1: _

predicted_iou: a learned head's guess at mask quality (trained to mimic real IoU, so it estimates confidence without ground truth).

stability_score: how consistent the mask stays when the threshold is nudged slightly — measures boundary robustness, not accuracy._

- Q2.2: _

SAM only finds where; a separate model (e.g. CLIP or Grounding DINO) is needed to say what each mask is, turning it into a semantic segmenter._

## Part 3 — Visualize & filter masks  *(15 pts)*

In [ ]:
import supervision as sv
# TODO: overlay all automatic masks on the image (source vs segmented side by side).

mask_annotator = sv.MaskAnnotator(color_lookup=sv.ColorLookup.INDEX)
detections = sv.Detections.from_sam(sam_result=sam_result)

annotated_image = mask_annotator.annotate(scene=image_bgr.copy(), detections=detections)

sv.plot_images_grid(
    images=[image_bgr, annotated_image],
    grid_size=(1, 2),
    titles=["source", "segmented"]
)

In [ ]:
# TODO: show the individual masks, largest-area first.

masks_sorted = sorted(sam_result, key=lambda x: x['area'], reverse=True)
mask_arrays = [m['segmentation'] for m in masks_sorted]

sv.plot_images_grid(
    images=mask_arrays[:16],  
    grid_size=(4, 4),
    titles=[f"area={m['area']}" for m in masks_sorted[:16]]
)

In [ ]:
# TODO: keep only masks with predicted_iou > 0.9 AND stability_score > 0.9;
# report how many survive and visualize them.

filtered_result = [m for m in sam_result if m['predicted_iou'] > 0.9 and m['stability_score'] > 0.9]
print(f"Survived: {len(filtered_result)} / {len(sam_result)}")

filtered_detections = sv.Detections.from_sam(sam_result=filtered_result)
filtered_annotated = mask_annotator.annotate(scene=image_bgr.copy(), detections=filtered_detections)

sv.plot_images_grid(
    images=[annotated_image, filtered_annotated],
    grid_size=(1, 2),
    titles=["all masks", "filtered masks"]
)

✍️ **Q3.1** How many masks survived, and what got removed? Would a stricter
or looser threshold be better for counting objects?

**Your answer:** _63/64 masks survived; only 1 mask was removed (whatever scored below 0.9 on either predicted_iou or stability_score which is typically a small or noisy-boundary fragment). Since nearly everything passed, the 0.9/0.9 threshold isn't doing much filtering here, for counting objects, this is probably too loose: SAM's automatic generator over-segments (e.g. separate masks for dog's ear, nose, leash, backpack pocket, logo patch) and a high confidence score doesn't mean "this is one whole object," so a stricter threshold alone won't fix over-counting. We'd also need area-based de-duplication (e.g. suppressing masks mostly contained inside a larger mask) to get an accurate object count._

## Part 4 — Box-prompted segmentation  *(20 pts)*

In [ ]:
import numpy as np
# TODO: mask_predictor = SamPredictor(sam); load dog.jpeg (RGB) and set_image(...).
mask_predictor = SamPredictor(sam)
image_bgr = cv2.imread(IMAGE_PATH)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
mask_predictor.set_image(image_rgb)


In [ ]:
# A box roughly covering the dog (x_min, y_min, x_max, y_max). Adjust to your image.
box = np.array([68, 247, 623, 925])

# TODO: predict from the box prompt with multimask_output=True.
masks, scores, logits =  mask_predictor.predict(box=box,multimask_output=True)   # masks -> (3, H, W)

assert masks is not None and masks.shape[0] == 3
print("candidate scores:", scores)


In [ ]:
# TODO: select the highest-scoring mask and visualize source vs segmented.

best_idx = np.argmax(scores)
best_mask = masks[best_idx]

detections = sv.Detections(
    xyxy=sv.mask_to_xyxy(masks=best_mask[np.newaxis, ...]),
    mask=best_mask[np.newaxis, ...]
)

mask_annotator = sv.MaskAnnotator(color_lookup=sv.ColorLookup.INDEX)
box_annotator = sv.BoxAnnotator(color_lookup=sv.ColorLookup.INDEX)

source_with_box = box_annotator.annotate(scene=image_bgr.copy(), detections=sv.Detections(xyxy=box[np.newaxis, ...]))
segmented_image = mask_annotator.annotate(scene=image_bgr.copy(), detections=detections)

sv.plot_images_grid(
    images=[source_with_box, segmented_image],
    grid_size=(1, 2),
    titles=["source (with box)", f"segmented (score={scores[best_idx]:.3f})"]
)


✍️ **Q4.1** Why does one box yield 3 masks with `multimask_output=True`, and
how did you choose one?

**Your answer:** _A single point or box is often ambiguous about scale like a box on the dog could mean "whole dog," "just the head," or "just the leash region." Since SAM can't know which one we want, multimask_output=True returns 3 candidates spanning different levels (typically: whole object, a sub-part, and a sub-sub-part), each with its own predicted_iou-style score. I chose the mask by taking np.argmax(scores) which is the candidate SAM is most confident matches the prompt . Though in ambiguous cases it's worth visually inspecting all 3, since the highest score doesn't always correspond to the segment we actually want._

## Part 5 — Point-prompted segmentation  *(15 pts)*

In [ ]:
# TODO: prompt SAM with positive point(s) on the object (point_labels=1) and
# visualize the best mask with the point(s) drawn on the image.

input_point = np.array([[350, 400]])   
input_label = np.array([1])           

masks, scores, logits = mask_predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True
)

best_idx = np.argmax(scores)
best_mask = masks[best_idx]

detections = sv.Detections(
    xyxy=sv.mask_to_xyxy(masks=best_mask[np.newaxis, ...]),
    mask=best_mask[np.newaxis, ...]
)

mask_annotator = sv.MaskAnnotator(color_lookup=sv.ColorLookup.INDEX)
segmented_image = mask_annotator.annotate(scene=image_bgr.copy(), detections=detections)

vis_image = segmented_image.copy()
cv2.circle(vis_image, tuple(input_point[0]), radius=10, color=(0, 255, 0), thickness=-1)

sv.plot_images_grid(
    images=[image_bgr, vis_image],
    grid_size=(1, 2),
    titles=["source", f"positive point (score={scores[best_idx]:.3f})"]
)


In [ ]:
# TODO: add a negative point (label 0) on the background/another object, re-run,
# and show how the mask changes.

input_point_neg = np.array([[350, 400], [150, 150]])   # a point on background
input_label_neg = np.array([1, 0])                

masks_neg, scores_neg, logits_neg = mask_predictor.predict(
    point_coords=input_point_neg,
    point_labels=input_label_neg,
    multimask_output=True
)

best_idx_neg = np.argmax(scores_neg)
best_mask_neg = masks_neg[best_idx_neg]

detections_neg = sv.Detections(
    xyxy=sv.mask_to_xyxy(masks=best_mask_neg[np.newaxis, ...]),
    mask=best_mask_neg[np.newaxis, ...]
)

segmented_image_neg = mask_annotator.annotate(scene=image_bgr.copy(), detections=detections_neg)

vis_image_neg = segmented_image_neg.copy()
cv2.circle(vis_image_neg, tuple(input_point_neg[0]), radius=10, color=(0, 255, 0), thickness=-1)
cv2.circle(vis_image_neg, tuple(input_point_neg[1]), radius=10, color=(0, 0, 255), thickness=-1)

sv.plot_images_grid(
    images=[vis_image, vis_image_neg],
    grid_size=(1, 2),
    titles=["positive only", f"+negative point (score={scores_neg[best_idx_neg]:.3f})"]
)

✍️ **Q5.1** What did the negative point do, and give a use-case where
negative points are essential.

**Your answer:** _The negative point tells SAM "this region should be excluded". It constrains the mask boundary away from that location, so if the positive point's mask was bleeding into a neighboring object (or background), adding a negative point there sharpens/shrinks the mask to stop at the correct edge.

Use case: segmenting a person standing in front of another person (or an object adjacent to our target) — a single positive point can produce a mask that merges both people if they're touching or similar in appearance/texture. Placing a negative point on the second person forces SAM to separate them, giving a clean mask of just the intended target._

## Part 6 — Quantitative evaluation vs ground truth (IoU)  *(15 pts)*

Prompt SAM with a **ground-truth box** from a labeled dataset and compute IoU
between SAM's mask and the ground-truth mask. Uses the Roboflow MRI brain-tumor
dataset (COCO). Needs a free Roboflow key; otherwise substitute your own
image+box+mask (still report a real IoU).

In [ ]:
# Provided plumbing: COCO dataclasses + helpers (do not modify).
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple, Optional
from dataclasses_json import dataclass_json
from supervision import Detections

@dataclass_json
@dataclass
class COCOCategory:
    id: int; name: str; supercategory: str

@dataclass_json
@dataclass
class COCOImage:
    id: int; width: int; height: int; file_name: str; license: int; date_captured: str
    coco_url: Optional[str] = None; flickr_url: Optional[str] = None

@dataclass_json
@dataclass
class COCOAnnotation:
    id: int; image_id: int; category_id: int
    segmentation: List[List[float]]; area: float
    bbox: Tuple[float, float, float, float]; iscrowd: int

@dataclass_json
@dataclass
class COCOLicense:
    id: int; name: str; url: str

@dataclass_json
@dataclass
class COCOJson:
    images: List[COCOImage]; annotations: List[COCOAnnotation]
    categories: List[COCOCategory]; licenses: List[COCOLicense]

def load_coco_json(json_file):
    import json
    with open(json_file, "r") as f:
        return COCOJson.from_dict(json.load(f))

class COCOJsonUtility:
    @staticmethod
    def get_annotations_by_image_id(coco_data, image_id):
        return [a for a in coco_data.annotations if a.image_id == image_id]
    @staticmethod
    def get_annotations_by_image_path(coco_data, image_path):
        image = COCOJsonUtility.get_image_by_path(coco_data, image_path)
        return COCOJsonUtility.get_annotations_by_image_id(coco_data, image.id) if image else None
    @staticmethod
    def get_image_by_path(coco_data, image_path):
        for image in coco_data.images:
            if image.file_name == image_path:
                return image
        return None
    @staticmethod
    def annotations2detections(annotations):
        class_id, xyxy = [], []
        for a in annotations:
            x, y, w, h = a.bbox
            class_id.append(a.category_id); xyxy.append([x, y, x + w, y + h])
        return Detections(xyxy=np.array(xyxy, dtype=int), class_id=np.array(class_id, dtype=int))


In [ ]:
# Provided (optional): download the Roboflow MRI brain-tumor dataset (COCO).
%cd {HOME}
import roboflow
from roboflow import Roboflow
roboflow.login()
rf = Roboflow()
project = rf.workspace("hashira-fhxpj").project("mri-brain-tumor")
dataset = project.version(1).download("coco")


In [ ]:
# Provided: load the test-split annotations.
DATA_SET_SUBDIRECTORY = "test"
ANNOTATIONS_FILE_PATH = os.path.join(dataset.location, DATA_SET_SUBDIRECTORY, "_annotations.coco.json")
coco_data = load_coco_json(ANNOTATIONS_FILE_PATH)
CLASSES = [c.name for c in coco_data.categories if c.supercategory != 'none']
IMAGES = [img.file_name for img in coco_data.images]
print("classes:", CLASSES, "| #images:", len(IMAGES))


In [ ]:
# TODO: mask_iou(mask_a, mask_b) -> intersection-over-union of two boolean masks.
def mask_iou(mask_a, mask_b):
    mask_a = mask_a.astype(bool)
    mask_b = mask_b.astype(bool)
    intersection = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    return intersection / union if union > 0 else 0.0

_a = np.zeros((10, 10), bool); _a[:5, :5] = True
_b = np.zeros((10, 10), bool); _b[:5, :] = True
assert abs(mask_iou(_a, _b) - 0.5) < 1e-6
print("mask_iou ok")

In [ ]:
# TODO: coco_seg_to_mask(segmentation, height, width) -> boolean HxW mask filled
# from the COCO polygon(s).
from PIL import Image as PILImage, ImageDraw

def coco_seg_to_mask(segmentation, height, width):
    mask_img = PILImage.new("L", (width, height), 0)
    draw = ImageDraw.Draw(mask_img)
    
    if not segmentation:
        return np.zeros((height, width), dtype=bool)

    for polygon in segmentation:
        if len(polygon) < 6:
            continue  
        
        pts = []
        for i in range(0, len(polygon), 2):
            x_val = polygon[i]
            y_val = polygon[i + 1]
            
            if 0.0 <= x_val <= 1.0 and 0.0 <= y_val <= 1.0:
                x_val *= width
                y_val *= height
                
            pts.append((x_val, y_val))
            
        draw.polygon(pts, outline=1, fill=1)
        
    return np.array(mask_img, dtype=bool)

In [ ]:
import random
random.seed(10)

# TODO: evaluate SAM on one labeled image:
#   pick an image, get its GT annotation, prompt SAM with the GT box, pick the best
#   mask, build the GT mask via coco_seg_to_mask, print IoU(SAM, GT) plus SAM's own
#   score, and visualize GT box | SAM mask | GT mask.

# an image with annotations
image_file = random.choice(IMAGES)
image_info = COCOJsonUtility.get_image_by_path(coco_data, image_file)
annotations = COCOJsonUtility.get_annotations_by_image_id(coco_data, image_info.id)

while not annotations:
    image_file = random.choice(IMAGES)
    image_info = COCOJsonUtility.get_image_by_path(coco_data, image_file)
    annotations = COCOJsonUtility.get_annotations_by_image_id(coco_data, image_info.id)

gt_annotation = annotations[0]

# Load image
image_path = os.path.join(dataset.location, DATA_SET_SUBDIRECTORY, image_file)
image_bgr = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# Ground Truth box (xyxy format)
x, y, w, h = gt_annotation.bbox
gt_box = np.array([x, y, x + w, y + h])

# Build GT mask via coco_seg_to_mask
gt_mask = coco_seg_to_mask(
    gt_annotation.segmentation, 
    image_info.height, 
    image_info.width, 
    bbox=gt_annotation.bbox
)

# Prompt SAM with GT box and pick best mask
mask_predictor.set_image(image_rgb)
masks, scores, logits = mask_predictor.predict(box=gt_box, multimask_output=True)

best_idx = np.argmax(scores)
sam_mask = masks[best_idx]
sam_score = scores[best_idx]

iou = mask_iou(sam_mask, gt_mask)

print(f"Image: {image_file}")
print(f"SAM self reported score: {sam_score:.4f}")
print(f"Measured IoU vs GT: {iou:.4f}")

# Visualizations: GT box | SAM mask | GT mask
box_annotator = sv.BoxAnnotator(color_lookup=sv.ColorLookup.INDEX)
mask_annotator = sv.MaskAnnotator(color_lookup=sv.ColorLookup.INDEX)

gt_box_vis = box_annotator.annotate(
    scene=image_bgr.copy(),
    detections=sv.Detections(xyxy=gt_box[np.newaxis, ...])
)

sam_mask_vis = mask_annotator.annotate(
    scene=image_bgr.copy(),
    detections=sv.Detections(
        xyxy=sv.mask_to_xyxy(masks=sam_mask[np.newaxis, ...]),
        mask=sam_mask[np.newaxis, ...]
    )
)

gt_mask_vis = mask_annotator.annotate(
    scene=image_bgr.copy(),
    detections=sv.Detections(
        xyxy=sv.mask_to_xyxy(masks=gt_mask[np.newaxis, ...]),
        mask=gt_mask[np.newaxis, ...]
    )
)

sv.plot_images_grid(
    images=[gt_box_vis, sam_mask_vis, gt_mask_vis],
    grid_size=(1, 3),
    titles=["GT box", f"SAM mask (IoU={iou:.3f})", "GT mask"]
)

✍️ **Q6.1 (Reflection)** Report your measured IoU vs SAM's self-reported
score. For medical tumor delineation, is zero-shot box-prompted SAM enough, or
would you train a dedicated model? Give two reasons.

### Bonus (+10)
- Mean IoU over all test images; does `predicted_iou` correlate with true IoU?
- Box-prompt vs point-prompt IoU on the same objects.
- `vit_b` vs `vit_h`: speed vs accuracy.

**Your answer / bonus notes:** __

Q6.1: SAM Evaluation for Tumor Delineation

* **Measured IoU vs. GT:** ~0.5545  
* **SAM Self-Reported Score:** 0.9751  

Conclusion:

**No, zero-shot box-prompted SAM is not sufficient for clinical tumor delineation.** A dedicated or fine-tuned model (e.g., MedSAM, nnU-Net) is required.

---

Key Reasons:

1. **Domain Misalignment & False Confidence:** SAM predicts high confidence (0.9751) based on natural image edges, but lacks domain-specific training on MRIs. It easily confuses infiltrative tumor boundaries or MRI artifacts with valid segment edges.
2. **Absence of Anatomical Priors:** Generic zero-shot models lack clinical priors regarding tissue geometry and pathology. Dedicated medical models trained with Dice/Focal loss are necessary to ensure the strict margin precision required for surgical resection or radiotherapy planning.

**Dataset Note:** The measured IoU (~0.55) is artificially low because the dataset contains bounding-box annotations, forcing evaluation against a rectangular GT mask rather than a true pixel-level boundary.

In [ ]:
#-Bonus: Mean IoU over all test images; does `predicted_iou` correlate with true IoU?

import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

records = []
for ann in coco_data.annotations:
    image_info = next(im for im in coco_data.images if im.id == ann.image_id)
    image_path = os.path.join(dataset.location, DATA_SET_SUBDIRECTORY, image_info.file_name)
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        continue
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    x, y, w, h = ann.bbox
    gt_box = np.array([x, y, x + w, y + h])
    gt_mask = bbox_to_mask(ann.bbox, image_info.height, image_info.width)

    mask_predictor.set_image(image_rgb)
    masks, scores, logits = mask_predictor.predict(box=gt_box, multimask_output=True)
    best_idx = np.argmax(scores)
    sam_mask = masks[best_idx]
    sam_score = scores[best_idx]

    iou = mask_iou(sam_mask, gt_mask)
    records.append((image_info.file_name, ann.id, sam_score, iou))

ious = np.array([r[3] for r in records])
pred = np.array([r[2] for r in records])

print(f"Evaluated {len(records)} annotations")
print(f"Mean IoU: {ious.mean():.4f} (std {ious.std():.4f})")

r, p = pearsonr(pred, ious)
print(f"Pearson corr(predicted_iou, true_iou): r={r:.3f}, p={p:.3f}")

plt.scatter(pred, ious)
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("SAM predicted_iou"); plt.ylabel("Measured IoU (box-as-mask)")
plt.title(f"r = {r:.2f}")
plt.show()

Why predicted_iou doesn't correlate with measured IoU here:

SAM's predicted_iou reflects how confident SAM is that its mask matches the object's true boundary — and on clean, high-contrast MRI tumors with a tight box prompt, that confidence is consistently high (0.95–1.0) regardless of which annotation we pick. But our "ground truth" here is a rectangle, not the tumor's actual shape. A tumor is roughly elliptical, so even a perfect segmentation can only overlap its bounding box up to a fixed ceiling (~78% for an ellipse, less for a loose box) — the rest is just background pixels inside the box that SAM correctly excludes. So the measured IoU variation (0.29–0.80) mostly reflects how tightly each box was drawn around its object, not how good SAM's segmentation is. Since those two things are independent, predicted_iou and box-relative IoU show no real correlation (r = −0.33, p = 0.271 — not significant with n=13). This isn't a flaw in SAM's confidence score; it's a mismatch between the metric (mask-vs-box) and what the confidence score is actually estimating (mask-vs-true-boundary).